STOCK PRICE PREDICTION MODEL USING LINEAR, RIDGE, LASSO, DECISION TREE, ELASTICNET, RANDOMFOREST, SVR

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import LinearSVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import xgboost as xgb

# 1. Data Preprocessing
df = pd.read_csv('NIFTY50.csv')

# Drop missing values
df = df.dropna().reset_index(drop=True)

# Feature Engineering: Extract date components
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek

# Feature Engineering: Create Lag features (previous 5 days)
for i in range(1, 6):
    df[f'Lag_{i}'] = df['Adj Close'].shift(i)

df = df.dropna().reset_index(drop=True)

# Define Features and Target
features = ['Year', 'Month', 'Day', 'DayOfWeek', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_5']
X = df[features]
y = df['Adj Close']

# Time-series split (Sequential)
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Scaler for distance-based models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {'Model': name, 'MAE': mae,  'RMSE': rmse, 'R-squared': r2}

results = []

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
results.append(evaluate(y_test, lr.predict(X_test_scaled), 'Linear Regression'))

# 2. Polynomial Regression (Degree 2)
poly_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])
poly_pipe.fit(X_train, y_train)
results.append(evaluate(y_test, poly_pipe.predict(X_test), 'Polynomial Regression'))

# 3. Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
results.append(evaluate(y_test, ridge.predict(X_test_scaled), 'Ridge Regression'))

# 4. Lasso Regression
lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)
results.append(evaluate(y_test, lasso.predict(X_test_scaled), 'Lasso Regression'))

# 5. Elastic Net Regression
en = ElasticNet(alpha=0.001, l1_ratio=0.5)
en.fit(X_train_scaled, y_train)
results.append(evaluate(y_test, en.predict(X_test_scaled), 'Elastic Net Regression'))

# 6. Decision Tree Regression
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
results.append(evaluate(y_test, dt.predict(X_test), 'Decision Tree Regression'))

# 7. Random Forest Regression
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
results.append(evaluate(y_test, rf.predict(X_test), 'Random Forest Regression'))

# 8. Support Vector Machine Regression (Linear SVR)
lsvr = LinearSVR(random_state=42, max_iter=10000)
lsvr.fit(X_train_scaled, y_train)
results.append(evaluate(y_test, lsvr.predict(X_test_scaled), 'SVR (Linear)'))

# 9. XGBoost Regression (using GradientBoostingRegressor as substitute)
# 9. XGBoost Regression


xgb_reg = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_reg.fit(X_train, y_train)
y_pred_xgb = xgb_reg.predict(X_test)
results.append(evaluate(y_test, y_pred_xgb, 'XGBoost Regression'))

# Display Results
results_df = pd.DataFrame(results)
print(results_df)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.923e+07, tolerance: 2.415e+06
  model = cd_fast.enet_coordinate_descent(


                      Model           MAE          RMSE  R-squared
0         Linear Regression    120.803112    164.451658   0.994758
1     Polynomial Regression    138.825437    200.209568   0.992231
2          Ridge Regression    124.801632    165.787047   0.994673
3          Lasso Regression    121.649447    164.858487   0.994732
4    Elastic Net Regression    126.504093    167.145485   0.994585
5  Decision Tree Regression   3325.157598   4013.330190  -2.121981
6  Random Forest Regression   3381.631542   4053.396136  -2.184627
7              SVR (Linear)  13836.029665  13983.234850 -36.899696
8        XGBoost Regression   3434.650142   4104.812373  -2.265931
